5 Oct. 2024

Using image labels to train object detection model on lego friends

References:
- https://cocodataset.org/#home (Coco model)
- https://www.robots.ox.ac.uk/~vgg/software/via/via.html (Labeling online software)
- https://www.kaggle.com/code/hmendonca/airbus-mask-rcnn-and-coco-transfer-learning (Coco transfer learning example)
- https://github.com/ultralytics/yolov5/issues/980 (Coco additional classes)
- ChatGPT (questions throughout the process)

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.9.18 | packaged by conda-forge | (main, Dec 23 2023, 16:36:46) 
[Clang 16.0.6 ]
/Users/mickey/anaconda3/envs/coco_python39_tf25/bin/python


In [2]:
import numpy as np
print(np.__version__)

1.20.2


In [3]:
import tensorflow as tf
print(tf.__version__)

2.5.0


# 1. Parse Json file for data we'll use

In [ ]:
import json

In [ ]:
# Loading Json file
with open('/content/drive/MyDrive/Colab Notebooks/via_project_24Sep2024_18h40m_json.json') as f:
  data = json.load(f)

# data.keys()

Example format for 1 image:

"lego-00.11.20-09.18.2024.jpg69105":{"filename":"lego-00.11.20-09.18.2024.jpg","size":69105,"regions":[{"shape_attributes":{"name":"rect","x":215,"y":200,"width":98,"height":122},"region_attributes":{"Name":"Lego Friend"}}],"file_attributes":{}}

In [ ]:
images = []
annotations = []

# Parsing through json file and identifying data stored for each image
for key, value in data.items():
  filename = value['filename']

  # There's only one region per image, but we'll use a for loop just to be sure
  for region in value['regions']:
    shape_attributes = region['shape_attributes']
    shape = shape_attributes['name']
    x = shape_attributes['x']
    y = shape_attributes['y']
    width = shape_attributes['width']
    height = shape_attributes['height']

    obj = region['region_attributes']['Name']

    # Calculate the bounding box coordinates
    xmin = x
    ymin = y
    xmax = x + width
    ymax = y + height

    images.append(filename)
    annotations.append({'bbox': [xmin, ymin, xmax, ymax],
                        'obj': obj})

In [ ]:
print(images)
print()
print(annotations)

['lego-00.11.20-09.18.2024.jpg', 'lego-01.06.20-09.18.2024.jpg', 'lego-02.15.20-09.18.2024.jpg', 'lego-03.10.20-09.18.2024.jpg', 'lego-03.10.20-09.18.2024.jpg', 'lego-04.13.20-09.18.2024.jpg', 'lego-05.11.20-09.18.2024.jpg', 'lego-05.14.20-09.18.2024.jpg', 'lego-06.07.20-09.18.2024.jpg', 'lego-06.07.20-09.18.2024.jpg', 'lego-07.08.20-09.18.2024.jpg', 'lego-07.08.20-09.18.2024.jpg', 'lego-07.12.20-09.18.2024.jpg', 'lego-07.12.20-09.18.2024.jpg', 'lego-07.17.20-09.18.2024.jpg', 'lego-07.17.20-09.18.2024.jpg', 'lego-08.06.20-09.18.2024.jpg', 'lego-09.05.20-09.18.2024.jpg', 'lego-09.05.20-09.18.2024.jpg', 'lego-09.09.20-09.18.2024.jpg', 'lego-09.16.20-09.18.2024.jpg', 'lego-09.16.20-09.18.2024.jpg', 'lego-11.04.20-09.18.2024.jpg', 'lego-12.17.20-09.18.2024.jpg', 'lego-13.02.20-09.18.2024.jpg', 'lego-13.14.20-09.18.2024.jpg', 'lego-13.14.20-09.18.2024.jpg', 'lego-14.05.20-09.18.2024.jpg', 'lego-14.11.20-09.18.2024.jpg', 'lego-15.15.20-09.18.2024.jpg', 'lego-16.16.20-09.18.2024.jpg', 'lego-1

In [ ]:
print(f"Number of images in list: {len(images)}")
print(f"Number of annotations in list: {len(annotations)}")

Number of images in list: 130
Number of annotations in list: 130


# 2. Preparing our Dataset
- Splitting data into training and testing sets

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_images, validate_images, train_annotations, validate_annotations = train_test_split(images, annotations, test_size=0.2, random_state=1)

In [ ]:
print(f"Number of training images: {len(train_images)} and number of training annotations: {len(train_annotations)}")
print(f"Number of validation images: {len(validate_images)} and number of validation annotations: {len(validate_annotations)}")

Number of training images: 104 and number of training annotations: 104
Number of validation images: 26 and number of validation annotations: 26


# 3. Convert Annotations to Tensorflow Format
- Want to use tf.data.Dataset, so we need to reformat our data to fit into this

In [ ]:
import tensorflow as tf

In [ ]:
def create_tf_data(image, annotations):
  """
  Takes the image and the annotations parsed from json file and converts it to an "example,"
  which is what tf.data.Dataset needs to be fed. We're serializing the data so we can put it into
  tf.data.Dataset and thus use it later.
  """
  image_path = f'/content/drive/MyDrive/lego_men_photos/{image}'
  image_string = tf.io.read_file(image_path)
  img = tf.image.decode_jpeg(image_string, channels=3)

  # Identifying the bounding box and object name in annotations
  bbx = annotations['bbox']
  obj_name = annotations['obj']

  # Creating tf.train.Example
  tf_example = tf.train.Example(features=tf.train.Features(feature={
      'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_string.numpy()])),
      # We appended data into bbox as 'bbox': [xmin, ymin, xmax, ymax], so we must label it in this order, too
      'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=[bbx[0]])),
      'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=[bbx[1]])),
      'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=[bbx[2]])),
      'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=[bbx[3]])),
      'image/object/class/text': tf.train.Feature(bytes_list=tf.train.BytesList(value=[obj_name.encode('utf8')])),
  }))

  return tf_example


# Using list comprehension to run all training data (images and annotations) through creat_df_data function
train_tfrecords = [create_tf_data(train_images[i], train_annotations[i]) for i in range(len(train_images))]

# Using list comprehension to run all validation data through, too
validate_tfrecords = [create_tf_data(validate_images[i], validate_annotations[i]) for i in range(len(validate_images))]

# Write training examples to TFRecord files
with tf.io.TFRecordWriter('training.record')as writer:
  for example in train_tfrecords:
    writer.write(example.SerializeToString())

# Write validation examples to TFRecord files
with tf.io.TFRecordWriter('validation.record') as writer:
  for example in validate_tfrecords:
    writer.write(example.SerializeToString())


### Coding Notes
- It's considered best practice to have a comma at the end of a value-key pair, since it makes it easier to add / remove lines later manually and with version control
- We use value=[bbx[3]] and not value=bbx[3] because "Tensorflow FloatList expects its input to be a list," according to ChatGPT.

Now we have the training and validation records to use later with Tensorflow.

# 4. Model Training
(summary of next instructions from ChatGPT)
- Using TensorFlow's Object Detection API
- Must choose a model (here we'll use Coco)
- Edit config file
  - Set paths for training and validation files (we just made these)
  -   Adjust the number of classes (here we have 1: "Lego Friend")
  - Set batch size, learning rate, ect.

## Download Tensorflow and Tensorflow Object Detection API
- Using Colab like a terminal now
  - Use a lot of "!"'s to signify that we're typing bash/command terminal commands

https://github.com/tensorflow/models.git = TensorFlow Model Garden (can pick models out to use from here)
- We'll be using the API

In [4]:
# Cloning TensorFlow models repo from github (including API)
!git clone https://github.com/tensorflow/models.git

Cloning into 'models'...
remote: Enumerating objects: 98298, done.
remote: Counting objects: 100% (811/811), done.
remote: Compressing objects: 100% (436/436), done.
remote: Total 98298 (delta 427), reused 684 (delta 355), pack-reused 97487 (from 1)
Receiving objects: 100% (98298/98298), 621.70 MiB | 7.67 MiB/s, done.
Resolving deltas: 100% (71415/71415), done.
Updating files: 100% (3874/3874), done.


## Protobuf Installation & Compilation

Reference for a lot of the code here:
https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/install.html

In [6]:
# Installing the obj. detection API
#!apt-get install protobuf-compiler # Works on Google Colab but not mac
!pip install protobuf-compiler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 8.0 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 5.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for colorama: filename=colorama-0.3.3-py3-none-any.whl size=14318 sha256=1bc8c9555381c814b5f28204f0872bc109b80898455439dffd8f876a77ee6396
  Stored in directory: /Users/mickey/Library/Caches/pip/wheels/ad/3c/d3/89e8fbc8dee7a115d06af19c8c5fda85cfeebc0dba676202d0
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [102 lines of output]
      /private/var/folders/9k/5h0j9gh93pb9ldsyknc0c9h80000gn/T/pip-install-h_z9oodj/grpcio_4fa3ef88da4f439bbfb46b776235fcfd/setup.py:21: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
        import pkg_resou

In [7]:
# Changing directory to models/research && downloading and compiling the protobuf libraries
!cd models/research && protoc object_detection/protos/*.proto --python_out=.
# !cp object_detection/packages/tf2/setup.py .
# !python -m pip install .

## COCO API Installation
- Still following instructions from above link
- COCO API: https://github.com/cocodataset/cocoapi.git

In [8]:
# Downloading cocoapi
!git clone https://github.com/cocodataset/cocoapi.git

Cloning into 'cocoapi'...
remote: Enumerating objects: 975, done.
remote: Total 975 (delta 0), reused 0 (delta 0), pack-reused 975 (from 1)
Receiving objects: 100% (975/975), 11.72 MiB | 8.42 MiB/s, done.
Resolving deltas: 100% (576/576), done.


In [10]:
!pip install pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 5.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.2/21.2 MB 5.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 6.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 6.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.19.5
    Uninstalling numpy-1.19.5:
      Successfully uninstalled numpy-1.19.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.9.1 requires numpy<1.25.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
tensorboard 2.11.2 requires protobuf<4,>=3.9.2, but you have protobuf 4.21.12 which is incompatible.
tensorflow 2.5.0 requires numpy~=1.19.2, but you have numpy 2.0.2 which is incompatible.


In [12]:
# Checking to make sure packages not changed in environment
print(sys.version)
print(np.__version__)
print(tf.__version__)

3.9.18 | packaged by conda-forge | (main, Dec 23 2023, 16:36:46) 
[Clang 16.0.6 ]
1.20.2
2.5.0


In [14]:
# Making the cocoapi repo
!cd cocoapi/PythonAPI && make

python setup.py build_ext --inplace
running build_ext
building 'pycocotools._mask' extension
clang -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /Users/mickey/anaconda3/envs/coco_python39_tf25/include -fPIC -O2 -isystem /Users/mickey/anaconda3/envs/coco_python39_tf25/include -I/Users/mickey/anaconda3/envs/coco_python39_tf25/lib/python3.9/site-packages/numpy/_core/include -I../common -I/Users/mickey/anaconda3/envs/coco_python39_tf25/include/python3.9 -c ../common/maskApi.c -o build/temp.macosx-10.9-x86_64-cpython-39/../common/maskApi.o -Wno-cpp -Wno-unused-function -std=c99
clang -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /Users/mickey/anaconda3/envs/coco_python39_tf25/include -fPIC -O2 -isystem /Users/mickey/anaconda3/envs/coco_python39_tf25/include -I/Users/mickey/anaconda3/envs/coco_python39_tf25/lib/python3.9/site-packages/numpy/_core/include -I../common -I/Users/mickey/anaco

In [19]:
# Making sure Cython is installed to help make
!pip install cython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 6.2 MB/s eta 0:00:00a 0:00:01


In [20]:
!cd cocoapi/PythonAPI && python setup.py build_ext --inplace

running build_ext
Compiling pycocotools/_mask.pyx because it changed.
[1/1] Cythonizing pycocotools/_mask.pyx
/Users/mickey/anaconda3/envs/coco_python39_tf25/lib/python3.9/site-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /Volumes/Dylan/Lego_Coco_training/cocoapi/PythonAPI/pycocotools/_mask.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
building 'pycocotools._mask' extension
clang -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /Users/mickey/anaconda3/envs/coco_python39_tf25/include -fPIC -O2 -isystem /Users/mickey/anaconda3/envs/coco_python39_tf25/include -I/Users/mickey/anaconda3/envs/coco_python39_tf25/lib/python3.9/site-packages/numpy/_core/include -I../common -I/Users/mickey/anaconda3/envs/coco_python39_tf25/include/python3.9 -c ../common/maskApi.c -o build/temp.macosx-10.9-x86_64-cpython-39/../comm

In [17]:
# Checking if clang is the issue (I don't think it is)
!cd cocoapi/PythonAPI && clang --version

Apple clang version 14.0.0 (clang-1400.0.29.202)
Target: x86_64-apple-darwin21.6.0
Thread model: posix
InstalledDir: /Library/Developer/CommandLineTools/usr/bin


In [23]:
!cp -r cocoapi/PythonAPI/pycocotools models/research/
# !cp -r cocoapi /content/models/research/

## Install Object Detection API
- Still following instructions in above link

In [27]:
# !cd models/research && protoc object_detection/protos/*.proto --python_out=.
# !cp object_detection/packages/tf2/setup.py .
# !python -m pip install .
!cd models/research && cp object_detection/packages/tf2/setup.py . && python -m pip install .

#!python -m pip install . # Doesn't like this because it forgets we're in models/research! Put in above line!

Processing /Volumes/Dylan/Lego_Coco_training/models/research
  Preparing metadata (setup.py) ... done
  Using cached avro-python3-1.10.2.tar.gz (38 kB)
  Preparing metadata (setup.py) ... done
  Using cached apache_beam-2.60.0-cp39-cp39-macosx_10_9_x86_64.whl.metadata (7.6 kB)
  Using cached lxml-5.3.0-cp39-cp39-macosx_10_9_x86_64.whl.metadata (3.8 kB)
  Using cached contextlib2-21.6.0-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached tf_slim-1.1.0-py2.py3-none-any.whl.metadata (1.6 kB)
  Using cached lvis-0.5.3-py3-none-any.whl.metadata (856 bytes)
  Using cached pandas-2.2.3-cp39-cp39-macosx_10_9_x86_64.whl.metadata (89 kB)
  Using cached tf_models_official-2.17.0-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached tensorflow_io-0.37.1-cp39-cp39-macosx_10_14_x86_64.whl.metadata (14 kB)
  Using cached keras-3.6.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pyparsing-2.4.7-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached sacrebleu-2.2.0-py3-none-any.whl.metadata (55 kB

In [29]:
# Testing installation
# From within /models/research/
!cd models/research/ && python object_detection/builders/model_builder_tf2_test.py

2024-10-23 14:23:20.785933: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/Volumes/Dylan/Lego_Coco_training/models/research/object_detection/builders/model_builder_tf2_test.py", line 24, in <module>
    from object_detection.builders import model_builder
  File "/Users/mickey/anaconda3/envs/coco_python39_tf25/lib/python3.9/site-packages/object_detection/builders/model_builder.py", line 26, in <module>
    from object_detection.builders import hyperparams_builder
  File "/Users/mickey/anaconda3/envs/coco_python39_tf25/lib/python3.9/site-packages/object_detection/builders/hyperparams_builder.py", line 27, in <module>
    from object_detection.core import freezable_sync_batch_norm
  File "/Users/mickey/anaconda3/

In [30]:
!pip install --upgrade tf_slim
!pip install tensorflow-object-detection-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.4/577.4 kB 3.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.4/587.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.6/694.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 7.3 MB/s eta 0:00:00a 0:00:01
  Created wheel for tensorflow-object-detection-api: filename=tensorflow_object_detection_api-0.1.1-py3-none-any.whl size=844489 sha256=c3bbc40004e3b99ec910ff490a3e713d6592ee651504073184309d4c1d01c067
  Stored in directory: /Users/mickey/Library/Caches/pip/wheels/38/6f/73/3c0c70f9a8219c31c96c495b686af1416154b60707fd96a4cd
Successfully built tensorflow-object-detection-api


In [32]:
# Add libraries to python path
import sys
sys.path.append('models/research')
sys.path.append('models/research/slim')

## Reaccess Dataset Preparation
- What we already wrote to the record files

In [33]:
!pwd

/Volumes/Dylan/Lego_Coco_training


In [34]:
training_record = '/training.record'
validation_record = '/validation.record'

In [37]:
# Making sure these files exist
!head -n 5 training.record
!head -n 5 validation.record

b     �>�
޾
*
image/object/class/text

d�      Z���
��
*
image/object/class/text



## Configure the Model
- Download pretrained model: Coco SSD MobileNet V2 for now
- Pick from TF Zoo: https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md
- Edit configuration file (.config)
  - Paths for training and validation datasets
  - Adjust the number of classes / batch size / ect.

https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/training.html

## Reconfiguring once more (Manually downloading the model this time)
- After downloading and unzipping in Google drive, downloaded as a zip file and unzipped on Dylan. 

In [40]:
#!cd /content/drive/MyDrive/ && tar -xvf ssd_mobilenet_v2_320x320_coco17_tpu-8.tar.gz

### Configuring the Training Pipeline
- Following link instructions still

In [39]:
!ls

LegoTensorflow_Dylan.ipynb
cocoapi
models
ssd_mobilenet_v2_320x320_coco17_tpu-8
ssd_mobilenet_v2_320x320_coco17_tpu-8-20241023T184513Z-001.zip
training.record
validation.record


In [44]:
# Copy the pipeline file for this model to the content folder
!cp ssd_mobilenet_v2_320x320_coco17_tpu-8/pipeline.config /pipeline.config
# !cp /pipeline.config /content/pipeline.config

cp: /pipeline.config: Read-only file system


Now we need to change the actual file...
1. Opening file in Google Colab
2. [line 3] num_classes = 1 (We only have "Lego Friend")
3. [line 138] batch_size = 8 (We can increase/decrease based on memory space later)
4. [line 162] fine_tune_checkpoint: "/content/drive/MyDrive/ssd_mobilenet_v2_320x320_coco17_tpu-8/checkpoint/ckpt-0"
  - DON'T INCLUDE FILE EXTENTIONS
  - Tensorflow will go through and look at all files with the ckpt-0 file name later
    - More flexibility and abiding by convention
    - Explained by ChatGPT
5. [line 168] fine_tune_checkpoint_type: "detection"
  - We want it to be training the whole detection model
6. [line 172] label_map_path: "/content/drive/MyDrive/label_map_legofriend.pbtxt"
7. [line 174] input_path: "/content/validation.record"
8. [line 182] label_map_path: "/content/drive/MyDrive/label_map_legofriend.pbtxt"
9. [line 186] input_path: "/content/validation.record"

### Making a label_map.pbtxt file
- We only have one class (Lego Friend), so we only need one item in our label class
- This is what our label map file looks like (Created with ChatGPT's help ['cause I didn't understand what the heck this was before I asked it, lol])


```
item {
  id: 1
  name: 'Lego Friend'
}
```



# Training the Model
Reference: https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/training.html


1. Want to use the /content/models/research/object_detection/model_main_tf2.py script from TensorFlow to train our model
  - Copy into our training folder (which for me is the /content file)
2. From the directory this file is in (/content), run the command:
```
python model_main_tf2.py --model_dir=models/my_ssd_resnet50_v1_fpn --pipeline_config_path=models/my_ssd_resnet50_v1_fpn/pipeline.config
```
  - WAIT for it to run...no touchy!

  
  ![no touchy](https://meme-generator.com/wp-content/uploads/mememe/2020/03/mememe_d75ef82eed2390ac48ea8a2a472f4a59-1.jpg)



### The model_dir here
- ALL INFO ABOUT MODEL USED IS IN PIPELINE FILE
- The model_dir here means where the training will store the checkpoints
- Create new directory for this: /content/drive/MyDrive/model_dir_1

In [ ]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.17.0
Uninstalling tensorflow-2.17.0:
  Successfully uninstalled tensorflow-2.17.0


In [ ]:
!python3 --version

Python 3.10.12


In [ ]:
!pip install tensorflow<2.8

/bin/bash: line 1: 2.8: No such file or directory


In [ ]:
!pip uninstall -y tensorflow-object-detection-api
!pip install tf-slim
!pip install -U tensorflow-object-detection-api

Found existing installation: tensorflow_object_detection_api 0.1.1
Uninstalling tensorflow_object_detection_api-0.1.1:
  Successfully uninstalled tensorflow_object_detection_api-0.1.1
  Using cached tensorflow_object_detection_api-0.1.1-py3-none-any.whl


In [ ]:
# Copying file into workspace
!cp /content/models/research/object_detection/model_main_tf2.py /content/drive/MyDrive/model_main_tf2.py

In [ ]:
# Running the long training prompt and filling in values
# python model_main_tf2.py --model_dir=models/my_ssd_resnet50_v1_fpn --pipeline_config_path=models/my_ssd_resnet50_v1_fpn/pipeline.config
!cd /content/drive/MyDrive/ && python model_main_tf2.py --model_dir=/content/drive/MyDrive/model_dir_1 --pipeline_config_path=pipeline.config --alsologtostderr
# Added --alsologtostderr because ChatGPT said this would show logs in the terminal

2024-10-05 21:22:54.775886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-05 21:22:54.837149: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-05 21:22:54.859114: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-05 21:22:54.895979: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-05 21:22:58.488502: W tensorflow/compiler/tf2

# Exporting Trained Model
Instructions from ChatGPT

```
!python models/research/object_detection/exporter_main_v2.py \
    --input_type=image_tensor \
    --pipeline_config_path=/content/pipeline.config \
    --trained_checkpoint_dir=/content/model_dir \
    --output_directory=/content/exported-model

```
- According to my understanding from what ChatGPT is saying, I shouldn't have to create any of these directories; training will create them
- Training will create checkpoints throughout training, and that's what it's referencing with trained_checkpoint_dir


In [ ]:
# Exporting, changed pipeline path
!python /content/models/research/object_detection/exporter_main_v2.py \
    --input_type=image_tensor \
    --pipeline_config_path=/content/drive/MyDrive/pipeline.config \
    --trained_checkpoint_dir=/content/drive/MyDrive/model_dir_1 \
    --output_directory=/content/drive/MyDrive/exported-model

# Converting to TensorFlow Lite
Instructions from ChatGPT


```
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model('/content/exported-model/saved_model')
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

```



# Downloading the Model
Instructions from ChatGPT


```
from google.colab import files

files.download('model.tflite')

```



Next steps according to ChatGPT

    Edit the Configuration File: You need to edit the .config file corresponding to the model you downloaded. You can use sed or any text editor to make the necessary changes. Here's an example:

python

!cp ssd_mobilenet_v2_coco_2018_03_29/pipeline.config /content/pipeline.config

# Edit the pipeline.config file to set paths and hyperparameters
!sed -i 's|PATH_TO_BE_CONFIGURED|/content|g' /content/pipeline.config

Make sure to set the paths for your training and validation datasets and adjust the number of classes, batch size, etc.
Step 5: Train the Model

Run the training script using the following command:

python

!python models/research/object_detection/model_main_tf2.py \
    --pipeline_config_path=/content/pipeline.config \
    --model_dir=/content/model_dir \
    --alsologtostderr \
    --num_steps=5000  # Adjust the number of steps as needed

Step 6: Export the Trained Model

Once training is complete, you can export the trained model to a TensorFlow SavedModel format or directly to TensorFlow Lite:

python

!python models/research/object_detection/exporter_main_v2.py \
    --input_type=image_tensor \
    --pipeline_config_path=/content/pipeline.config \
    --trained_checkpoint_dir=/content/model_dir \
    --output_directory=/content/exported-model

Step 7: Convert to TensorFlow Lite (Optional)

If you want to convert the model to TensorFlow Lite:

python

import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model('/content/exported-model/saved_model')
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

Step 8: Download the Model

Finally, you can download your model or any files created during the process:

python

from google.colab import files

files.download('model.tflite')